# 02 — Inventory & Calculation

**Rules:** PEP ecopassport PCR ed4 (S08) and PSR-0018 ed1.3 (S01), as set in `01_scope_boundary`.

This notebook turns the decisions in notebook 01 into numbers: the life cycle emissions of one wallbox (declared unit, for CDR sizing), the emissions per kWh supplied (functional unit, for PEP benchmarking), and the share of each stage (hotspots).

**How to use:** Fill in every cell marked `TODO`, in order. Every number must have a source ID (`data/raw/SOURCES.md`) or an assumption ID (`docs/assumptions_log.md`). Calculation cells produce results only after the inputs above them are filled in.

## 0. Parameters from notebook 01

These values were decided in notebook 01. Do not change them here; change notebook 01 first if a decision changes.

In [8]:
SERVICE_LIFE_YEARS = 10                 # S01, p. 20
ENERGY_PER_CHARGING_POINT_KWH = 28_251  # S01, p. 20 (derivation in notebook 01, Section 2)
CHARGING_POINTS = 1                     # Notebook 01, reference flow
CABLE_LENGTH_FT = 24                    # A02 (sensitivity range 13.1-25 ft)
FT_TO_M = 0.3048                        # Exact unit conversion
CUTOFF_THRESHOLD = 0.05                 # S08, §2.3 (aggregate mass, energy, impact)
DQR_RELEVANCE_SHARE = 0.80              # S08, §2.9.3 (score datasets covering >= 80% of impact)
DEFAULT_DQR = 5                         # S08, §2.9.3 (when quality cannot be assessed)

ENERGY_SUPPLIED_KWH = ENERGY_PER_CHARGING_POINT_KWH * CHARGING_POINTS
REFERENCE_FLOW = 1 / ENERGY_SUPPLIED_KWH  # wallboxes per kWh (functional unit)
CABLE_LENGTH_M = CABLE_LENGTH_FT * FT_TO_M

print(f"Energy supplied over service life: {ENERGY_SUPPLIED_KWH:,} kWh")
print(f"Reference flow: {REFERENCE_FLOW:.4e} wallboxes per kWh")
print(f"Cable length: {CABLE_LENGTH_M:.2f} m")

Energy supplied over service life: 28,251 kWh
Reference flow: 3.5397e-05 wallboxes per kWh
Cable length: 7.32 m


## 1. Data Sources

**Guiding questions**
- Which public databases or datasets can provide emission factors for the materials in a wallbox? (ecoinvent requires a paid license, so it cannot be used.)
- Where does each candidate rank in the PCR ed4 source priority order (S08, §2.9)?
- Which candidate offers global average (GLO) datasets, as required by A03?

In [9]:
import pandas as pd

data_sources = pd.DataFrame(
    [
        {
            "source_id": "S09",
            "name": "Copper and Copper Alloy Semi-Fabricated Products LCA",
            "publisher": "International Copper Association (ICA)",
            "public_access": True,
            "pcr_priority_level": 4,
            "glo_datasets": "Global (excludes China)",
            "notes": "Copper wire rod; ends before wire drawing, which must be added separately",
        },
    ]
)
data_sources

,source_id,name,publisher,public_access,pcr_priority_level,glo_datasets,notes
0,S09,Copper and Copper Alloy Semi-Fabricated Produc...,International Copper Association (ICA),True,4,Global (excludes China),"Copper wire rod; ends before wire drawing, whi..."


## 2. Reference Product and Bill of Materials (BOM)

With public data only, the BOM is a **proxy BOM** built with a data hierarchy:

| Tier | Data | Use in this project |
|---|---|---|
| 1 | Manufacturer's actual BOM | Not available (public data only) |
| 2 | Material composition from published PEPs or EPDs of similar wallboxes | Main source of material shares |
| 3 | Engineering estimate | Cable copper from length × gauge |
| 4 | Published LCA studies of EV chargers | Fill remaining gaps |
| 5 | Spend-based estimate | Screening only, last resort |

**Guiding questions**
- Which reference product has a public total mass (S04–S06)? Does that mass include the cable and the mounting bracket?
- Which published PEP or EPD of a wallbox can provide material shares?
- Which PCR ed4 Table 2 items (copper, gold, silver, microprocessors) appear, and where?
- Does the sum of all components equal the total product mass (mass balance)?

### 2a. How the proxy BOM is built

No manufacturer BOM is public, so the BOM is assembled from four parts, each with its own source:

| Part | Mass | How it is derived |
|---|---|---|
| Body (everything except cable, connector, packaging) | 3.6 kg (A04) | Median of sources that state a mass without the cable; split into materials with the mean shares of five PEPs (S10–S14) |
| Charging cable | ≈ 3.0 kg (A05) | Copper from conductor size × length × copper density; jacket scaled from S13 |
| Connector (charging handle) | 0.54 kg (A06) | Manufacturer weight of a J1772 connector (S17); material split decided after emission factors are known |
| Packaging | 2.05 kg (A09) | Median of the five PEPs; split with their mean shares |

### 2b. Body: material shares from five PEPs (A04, A07, A08)

Each PEP lists materials for the whole product. To use them for the body only, the charging cable and packaging are removed and each PEP is converted to shares of its own body. The mean of the five PEPs is then applied to the 3.6 kg body.

Grouping rules:
- PC and ABS are combined as housing PC (A07).
- PCB = "electronic cards" (S10, S11), "PWB" (S12), "electronic components" (S14), epoxy + glass + TBBPA (S13). The S12 and S13 values may understate the PCB (A08).
- S13 does not report its cable separately, so all its copper and PVC are assigned to the cable (A05 note); its body copper is only the brass.

In [10]:
BODY_MASS_KG = 3.6  # A04

# Published body materials (cable and packaging excluded).
# S10, S11, S12, S14: % of total mass incl. packaging (p. 2); S13: grams (p. 2). Only the ratios are used.
pep_body = pd.DataFrame({
    "S10": {"PC": 25.0, "ABS": 1.4, "PA": 2.5, "PBT": 0.4, "Other plastics": 0.4 + 0.5, "Steel": 3.2,
            "Copper": 1.0, "Aluminium": 0.1, "PCB": 8.4, "Internal wiring": 1.7, "Other components": 0},
    "S11": {"PC": 21.7, "ABS": 1.2, "PA": 3.3, "PBT": 0.7, "Other plastics": 0.6 + 0.2, "Steel": 3.8 + 0.4,
            "Copper": 1.4, "Aluminium": 0, "PCB": 8.8, "Internal wiring": 0.8, "Other components": 0},
    "S12": {"PC": 18.0, "ABS": 0, "PA": 5.8, "PBT": 0.8, "Other plastics": 2.6 + 0.8 + 0.7, "Steel": 13.0 + 0.4,
            "Copper": 3.4, "Aluminium": 0.1, "PCB": 1.7 + 0.4, "Internal wiring": 0, "Other components": 1.9 + 0.1},
    "S13": {"PC": 3012.00, "ABS": 0, "PA": 0, "PBT": 0, "Other plastics": 36.71 + 36.03 + 102.68, "Steel": 68.51 + 52.08,
            "Copper": 40.38, "Aluminium": 35.17 + 115.54, "PCB": 133.13 + 207.79 + 20.98, "Internal wiring": 0, "Other components": 100.68},
    "S14": {"PC": 1.1, "ABS": 21.7, "PA": 5.6, "PBT": 0, "Other plastics": 0.2, "Steel": 1.6,
            "Copper": 1.7 + 0.1, "Aluminium": 0.4, "PCB": 10.7, "Internal wiring": 0, "Other components": 0},
}).T

body_shares = pep_body.div(pep_body.sum(axis=1), axis=0)   # each PEP as shares of its own body
mean_shares = body_shares.mean()                            # mean of the five PEPs
mean_shares["PC"] += mean_shares.pop("ABS")                 # A07: housing modeled as PC
body_materials = (mean_shares * BODY_MASS_KG).sort_values(ascending=False)

pd.DataFrame({"share": mean_shares.reindex(body_materials.index).round(3), "mass_kg": body_materials.round(3)})

,share,mass_kg
PC,0.555,2.000
PCB,0.155,0.558
Steel,0.101,0.365
PA,0.076,0.273
Copper,0.035,0.126
Other plastics,0.034,0.123
Other components,0.013,0.047
Internal wiring,0.011,0.041
Aluminium,0.010,0.037
PBT,0.008,0.030


### 2c. Charging cable (A05)

Copper mass = conductor cross-section × cable length × copper density, for each conductor:
- 2 power conductors at 8 AWG (S07), 1 ground at 10 AWG and 1 signal at 18 AWG (A05)
- Cross-sections are the standard AWG values; copper density is 8.96 g/cm³

Jacket and insulation = 204 g per metre (S13: 1,326.30 g of PVC on a 6.5 m cable) × cable length.

In [11]:
COPPER_DENSITY_KG_M3 = 8960                           # physical constant
AWG_AREA_MM2 = {8: 8.367, 10: 5.261, 18: 0.823}       # standard AWG cross-sections
CABLE_CONDUCTORS_AWG = [8, 8, 10, 18]                 # A05: 2 power (S07), 1 ground, 1 signal
JACKET_KG_PER_M = 1.32630 / 6.5                       # S13

cable_copper_kg = sum(AWG_AREA_MM2[g] * 1e-6 * CABLE_LENGTH_M * COPPER_DENSITY_KG_M3 for g in CABLE_CONDUCTORS_AWG)
cable_jacket_kg = JACKET_KG_PER_M * CABLE_LENGTH_M
print(f"Cable copper: {cable_copper_kg:.3f} kg | Jacket: {cable_jacket_kg:.3f} kg | Cable: {cable_copper_kg + cable_jacket_kg:.3f} kg")

Cable copper: 1.496 kg | Jacket: 1.493 kg | Cable: 2.988 kg


### 2d. Connector and packaging (A06, A09)

- Connector: 540 g (S17). Its plastic and copper split is not public; it stays as one row until the emission factors in Section 3 show which material is the conservative choice.
- Packaging: median mass of the five PEPs, split with their mean shares of cardboard, wood, and paper.

In [12]:
CONNECTOR_MASS_KG = 0.54  # A06, S17

# Packaging as published (same units as pep_body); S12 and S14 list no wood or no paper.
pep_packaging = pd.DataFrame({
    "S10": {"Cardboard": 19.6, "Wood": 6.6, "Paper": 0.8},
    "S11": {"Cardboard": 16.8, "Wood": 5.7, "Paper": 0.7},
    "S12": {"Cardboard": 10.4, "Wood": 10.7, "Paper": 1.8},
    "S13": {"Cardboard": 1352.86, "Wood": 183.57, "Paper": 0},
    "S14": {"Cardboard": 16.2, "Wood": 0, "Paper": 0.0},
}).T
PACKAGING_MASS_KG = pd.Series({"S10": 2.06, "S11": 2.05, "S12": 2.17, "S13": 1.54, "S14": 1.28}).median()  # A09
packaging_shares = pep_packaging.div(pep_packaging.sum(axis=1), axis=0).mean()
packaging_materials = packaging_shares * PACKAGING_MASS_KG
pd.DataFrame({"share": packaging_shares.round(3), "mass_kg": packaging_materials.round(3)})

,share,mass_kg
Cardboard,0.757,1.552
Wood,0.215,0.442
Paper,0.028,0.057


### 2e. BOM table

`data_tier` follows the table in Section 2: 2 = shares from published PEPs, 3 = engineering estimate or proxy product. `table2_item` marks PCR ed4 Table 2 materials (copper; the PCB contains gold, silver, and microprocessors).

In [13]:
rows = []
body_note = "3.6 kg body (A04) x mean share of five PEPs"
for material, mass in body_materials.items():
    rows.append({"component": "Body", "material": material, "mass_kg": mass,
                 "source_id": "S10-S14; A04", "data_tier": 2,
                 "table2_item": material in ("PCB", "Copper", "Internal wiring"),
                 "derivation": body_note + (" (PC + ABS, A07)" if material == "PC" else " (A08)" if material == "PCB" else "")})
rows += [
    {"component": "Charging cable", "material": "Copper", "mass_kg": cable_copper_kg, "source_id": "S07; A02; A05",
     "data_tier": 3, "table2_item": True, "derivation": "Cross-section x 7.32 m x 8.96 g/cm3 for 2x8, 1x10, 1x18 AWG"},
    {"component": "Charging cable", "material": "PVC", "mass_kg": cable_jacket_kg, "source_id": "S13; A02; A05",
     "data_tier": 3, "table2_item": False, "derivation": "204 g/m (S13 PVC / 6.5 m) x 7.32 m"},
    {"component": "Connector", "material": "Connector (material TBD)", "mass_kg": CONNECTOR_MASS_KG, "source_id": "S17; A06",
     "data_tier": 3, "table2_item": True, "derivation": "Manufacturer weight of a J1772 demo connector"},
]
for material, mass in packaging_materials.items():
    rows.append({"component": "Packaging", "material": material, "mass_kg": mass, "source_id": "S10-S14; A09",
                 "data_tier": 2, "table2_item": False, "derivation": "2.05 kg packaging (A09) x mean share of five PEPs"})

bom = pd.DataFrame(rows)
bom["mass_kg"] = bom["mass_kg"].round(3)
bom

,component,material,mass_kg,source_id,data_tier,table2_item,derivation
0,Body,PC,2.000,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs (P...
1,Body,PCB,0.558,S10-S14; A04,2,True,3.6 kg body (A04) x mean share of five PEPs (A08)
2,Body,Steel,0.365,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs
3,Body,PA,0.273,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs
4,Body,Copper,0.126,S10-S14; A04,2,True,3.6 kg body (A04) x mean share of five PEPs
5,Body,Other plastics,0.123,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs
6,Body,Other components,0.047,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs
7,Body,Internal wiring,0.041,S10-S14; A04,2,True,3.6 kg body (A04) x mean share of five PEPs
8,Body,Aluminium,0.037,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs
9,Body,PBT,0.030,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs


### 2f. Mass balance and plausibility

1. **Mass balance:** the BOM rows must add up to the component masses they were derived from.
2. **Plausibility:** the product mass is compared with independent figures. ChargePoint (S04) and Tesla (S05) totals are only a cross-check, because their datasheets do not state whether the total includes the cable.

In [14]:
product = bom[bom["component"] != "Packaging"]["mass_kg"].sum()
expected = BODY_MASS_KG + cable_copper_kg + cable_jacket_kg + CONNECTOR_MASS_KG
REFERENCE_PRODUCT_MASS_KG = round(expected, 3)  # product without packaging, used for the cut-off check in Section 9

print(f"BOM product total: {product:.3f} kg | Sum of component masses: {expected:.3f} kg | Gap: {product - expected:+.3f} kg")
print(f"BOM total incl. packaging: {bom['mass_kg'].sum():.2f} kg (five PEPs: 7.62-9.38 kg incl. packaging)")
print(f"Body + cable without connector: {BODY_MASS_KG + cable_copper_kg + cable_jacket_kg:.2f} kg "
      f"(ChargePoint 6.3 kg, S04; Tesla 6.8 kg incl. bracket, S05)")
bom.groupby("component")["mass_kg"].sum().round(3)

BOM product total: 7.129 kg | Sum of component masses: 7.128 kg | Gap: +0.001 kg
BOM total incl. packaging: 9.18 kg (five PEPs: 7.62-9.38 kg incl. packaging)
Body + cable without connector: 6.59 kg (ChargePoint 6.3 kg, S04; Tesla 6.8 kg incl. bracket, S05)


component
Body              3.600
Charging cable    2.989
Connector         0.540
Packaging         2.051
Name: mass_kg, dtype: float64

## 3. Emission Factors and Data Quality (DQR)

**Guiding questions**
- For each material in the BOM, which dataset gives its emission factor (kg CO₂e per kg)?
- How well does each dataset match in technology (TeR), geography (GR), and time (TiR), on a scale from 1 (best) to 5 (worst)?
- Which datasets together cover at least 80% of the climate change impact, so they must be scored?

In [15]:
# TODO: one row per material. Scores follow PCR ed4 Annex E (S08); use DEFAULT_DQR when quality cannot be assessed.
emission_factors = pd.DataFrame(
    columns=["material", "ef_kgco2e_per_kg", "dataset", "source_id", "TeR", "GR", "TiR"]
)

if not emission_factors.empty:
    emission_factors["DQR"] = emission_factors[["TeR", "GR", "TiR"]].mean(axis=1)
emission_factors

,material,ef_kgco2e_per_kg,dataset,source_id,TeR,GR,TiR


## 4. Manufacturing Stage (A1–A3)

Emissions = mass of each material × its emission factor.

In [16]:
manufacturing = pd.DataFrame()
if not bom.empty and not emission_factors.empty:
    manufacturing = bom.merge(emission_factors[["material", "ef_kgco2e_per_kg"]], on="material", how="left")
    missing = manufacturing[manufacturing["ef_kgco2e_per_kg"].isna()]["material"].unique()
    if len(missing):
        print("Materials without an emission factor:", list(missing))
    manufacturing["kgco2e"] = manufacturing["mass_kg"] * manufacturing["ef_kgco2e_per_kg"]
    print(f"Manufacturing total: {manufacturing['kgco2e'].sum():.2f} kg CO2e per wallbox")
manufacturing

""


## 5. Distribution (A4) and Installation (A5)

**Guiding questions**
- What default transport scenario does PCR ed4 set for distribution (S08)?
- PSR-0018 sets packaging waste treatment rates and a 1,000 km transport distance for installation (S01, pp. 17–18). How do they apply to the packaging in your BOM?

In [17]:
# TODO: calculate from the PCR / PSR default scenarios and record each input's source ID
DISTRIBUTION_KGCO2E = None
INSTALLATION_KGCO2E = None

## 6. Use Stage (B1–B7)

PSR-0018 (S01, §3.2.4) counts only the charger's own consumption; electricity delivered to the vehicle is excluded.

- Conduction losses = R × I² × T × X / 1000 (kWh)
- Intrinsic consumption = (P_active × %T_active + P_standby_active × %T_standby_active + P_standby_passive × %T_standby_passive + P_off × %T_off) × service life

**Guiding questions**
- Which inputs come from the PSR scenario (S01, p. 20; S02), and which must come from the reference product's datasheet?
- Which EPA eGRID emission factors (kg CO₂e per kWh) represent California and Texas, and which eGRID year?

In [18]:
# TODO: fill in each input with its source ID
R_OHM = None            # resistance of the conductive path
I_AMP = None            # charging current
T_HOURS = None          # charge time per session
X_CHARGES = None        # number of charges over the service life
INTRINSIC_KWH = None    # intrinsic consumption over the service life

GRID_EF_KGCO2E_PER_KWH = {"California": None, "Texas": None}  # EPA eGRID

use_stage = {}
if None not in (R_OHM, I_AMP, T_HOURS, X_CHARGES, INTRINSIC_KWH):
    losses_kwh = R_OHM * I_AMP**2 * T_HOURS * X_CHARGES / 1000
    use_kwh = losses_kwh + INTRINSIC_KWH
    print(f"Use-stage electricity: {use_kwh:.1f} kWh over {SERVICE_LIFE_YEARS} years")
    for state, ef in GRID_EF_KGCO2E_PER_KWH.items():
        if ef is not None:
            use_stage[state] = use_kwh * ef
use_stage

{}

## 7. End of Life (C1–C4)

**Guiding questions**
- What default end-of-life scenario does PCR ed4 set for electrical and electronic equipment (S08)?
- Which materials in the BOM are recycled, incinerated, or landfilled under that scenario?

In [19]:
# TODO: calculate from the PCR default end-of-life scenario and record each input's source ID
END_OF_LIFE_KGCO2E = None

## 8. Results

- **Per wallbox (declared unit):** sizes the CDR plan in notebook 04
- **Per kWh (functional unit):** benchmarks against published PEPs in notebook 03
- **Share by stage:** identifies the hotspots

In [20]:
stage_values = {
    "Manufacturing (A1-A3)": manufacturing["kgco2e"].sum() if "kgco2e" in manufacturing else None,
    "Distribution (A4)": DISTRIBUTION_KGCO2E,
    "Installation (A5)": INSTALLATION_KGCO2E,
    "End of life (C1-C4)": END_OF_LIFE_KGCO2E,
}

results = {}
if None not in stage_values.values():
    for state, use_kgco2e in use_stage.items():
        stages = {**stage_values, "Use (B1-B7)": use_kgco2e}
        total = sum(stages.values())
        results[state] = pd.DataFrame(
            {
                "kgco2e_per_wallbox": stages,
                "kgco2e_per_kwh": {k: v * REFERENCE_FLOW for k, v in stages.items()},
                "share": {k: v / total for k, v in stages.items()},
            }
        )
        print(f"{state}: {total:.1f} kg CO2e per wallbox | {total * REFERENCE_FLOW:.2e} kg CO2e per kWh")
results

{}

## 9. Cut-off Compliance

Notebook 01, Section 4: excluded flows must stay at or below 5% of mass, primary energy, and environmental impact in aggregate (S08, §2.3). PCR ed4 Table 2 items cannot be excluded.

**Guiding questions**
- Which items were left out of the BOM or the stage calculations, and why?
- How was each excluded item's mass, energy, and impact estimated?

In [21]:
# TODO: one row per excluded item
exclusions = pd.DataFrame(
    columns=["item", "stage", "mass_kg", "energy_mj", "kgco2e", "reason", "source_id"]
)

if not exclusions.empty and REFERENCE_PRODUCT_MASS_KG:
    mass_share = exclusions["mass_kg"].sum() / REFERENCE_PRODUCT_MASS_KG
    print(f"Excluded mass share: {mass_share:.1%} (threshold {CUTOFF_THRESHOLD:.0%})")
exclusions

,item,stage,mass_kg,energy_mj,kgco2e,reason,source_id


## 10. Consistency Check (before moving to notebook 03)

Review this notebook as if you were the assurance reviewer.

- [ ] The parameters in Section 0 match notebook 01.
- [ ] Every BOM row has a source ID and a data tier, and the mass balance gap is explained.
- [ ] All PCR ed4 Table 2 items in the product are included.
- [ ] Every emission factor has a source ID and a DQR score.
- [ ] Electricity delivered to the vehicle is excluded from the use stage.
- [ ] Excluded items stay within the 5% cut-off thresholds.
- [ ] Every number is either sourced in `data/raw/SOURCES.md` or logged in `docs/assumptions_log.md`.
- [ ] The README status has been updated.

## References

Full details (publisher, URL, version, access date) are in [`data/raw/SOURCES.md`](../data/raw/SOURCES.md). Assumptions are in [`docs/assumptions_log.md`](../docs/assumptions_log.md).

| ID | Source | Used for |
|---|---|---|
| S01 | P.E.P. Association, PSR-0018 Electric Vehicle Charging Infrastructures, ed1.3 (2025-04-08) | Service life, energy supplied, use-stage formulas, installation defaults |
| S08 | P.E.P. Association, PEP-PCR ed4 (2021-09-06) | Cut-off threshold, DQR method |

<!-- TODO: add each source as it is used in this notebook -->